<font color='IndianRed'>
<font size=9> 

**Curso Inferencia Estadística**

<font color = 'DodgerBlue'>
<font size = 5>
    
**Diagnósticos de regresión.**

<font color = 'Black'>
<font size = 3>

Para que el modelo de **regresión lineal** sea fiable, en las conclusiones derivadas de las **estimaciones** de **intervalos** e **inferencias** (**intervalos de confianza**, **contrastes de hipótesis**,...) que realizamos a partir de dicho modelo se tienen que verificar cietas hipótesis.

Las tareas que realizan dichas verificaciones se denominan **diagnósticos de regresión**.

Estos se clasifican en tres categorías:

* Errores: los errores deben seguir una $N(0,\sigma)$ con la misma **varianza** y ser incorrelacionados.


* Modelo: los puntos se tienen que ajustar a la estructura lineal considerada.


* Observaciones anómalas: a veces unas cuántas observaciones no se ajustan al modelo y hay que detectarlas.


En general hay dos métodos: gráficos y numéricos. 

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Normalidad de los residuos.**

<font color = 'Black'>
<font size = 3>

Para detectar la **normalidad** de los **residuos**, podemos aplicar todos los tests de **normalidad** vistos en el tema de **Bondad de Ajuste**:

* Test **Kolmogorov-Smirnov-Lilliefors**.

* Test **Anderson-Darling**.

* Test de **Shapiro-Wilks**.

* Test **D'Agostino-Pearson**.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Normalidad de los residuos en Python**

<font color = 'Black'>
<font size = 3>
    
En **Python**, si ``modelo`` es el modelo, entonces ``modelo.resid`` es el vector de errores. Por lo tanto es a este a vector a quien le tenemos que hacer la prueba de gaussianidad. Recuerda que para utilizarlos en **Python** requieres la biblioteca ``scipy.stats`` y/o ``statsmodels``

$p$-valor|Decisión|Significado
:--|:--|:--
Pequeño|Rechazar $H_0$|Hay buena probabilidad de que **NO es gaussiana**
Grande|Rechazar $H_1$|Hay buena probabilidad de que **SÍ es gaussiana**

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Homocedasticidad.**

<font color = 'Black'>
<font size = 3>
    
Para verificar la **homocedasticidad** de los residuos, podemos aplicar el **Test de Breuch-Pagan**. Su aplicación es bastante sencilla:

* Obtener los residuos $\{e_i\}_{i=1}^n$ de la **regresión lineal** inicial.


* Calcular el **coeficiente de determinación** $R^2$ de la **regresión lineal** de los $e_i^2$ respecto de las variables iniciales. Es decir, calcular el **coeficiente de determinación** del modelo siguiente:

$$E^2=\gamma_0+\gamma_1X_1+\cdots+\gamma_kX_k$$

Se busca escribir a los errores $E^2$ como **regresión lineal** de las columnas explicativas.

* Calcular el **estadístico** $X_0=nR^2$ el cual, suponiendo **homocedasticidad**, sigue una $\chi^2_k$.


* Calculamos el p-valor $P(\chi^2_k\ge X_0)$ con el significado usual.


La regla de decisión es 

$p$-valor|Decisión|Significado
:--|:--|:--
Pequeño|Rechazar $H_0$|Hay buena probabilidad de que los residuos **NO sean homocedásticos**
Grande|Rechazar $H_1$|Hay buena probabilidad de que los residuos **SÍ sean homocedásticos**

También se puede hacer una prueba gráfica: cuando se grafica la nube de puntos de los valores ajustados (es decir, las $\hat{y_i}$) vs los residuos, deberemos observar un **cielo estrellado** centrado en la recta $y=0$.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Homocedasticidad en Python**

<font color = 'Black'>
<font size = 3>

Para la prueba gráfica:

```python
################################################################
########################    EN PYTHON    #######################
################################################################

from plotnine import *

#### Obtener los residuos
residuos = modelo.resid

#### Obtener los valores y_gorro
y_gorro = modelo.fittedvalues

#### Realizar el gráfico

var_errores = pd.DataFrame({"y_gorro":y_gorro, "residuos":residuos})

(ggplot(data=var_errores) + 
 geom_point(mapping=aes(x="y_gorro",y="residuos"),color="blue") +
 geom_hline(yintercept=0)
)

################################################################
```

Para la prueba de  Breuch-Pagan en Python, simplemente usamos ``het_breuschpagan`` de la paquetería ``statsmodels.stats.diagnostic``.

```python
################################################################
########################    EN PYTHON    #######################
################################################################

from statsmodels.stats.diagnostic import het_breuschpagan

# Realizar la prueba de Breusch-Pagan

bp_test = het_breuschpagan(residuos, X)

# Los resultados
bp_stat = bp_test[0]   # Estadístico de la prueba
bp_p_value = bp_test[1]  # p-valor asociado

print(f"Estadístico de Breusch-Pagan: {bp_stat}")
print(f"p-valor: {bp_p_value}")

# Interpretación
if bp_p_value > 0.05:
    print("No se rechaza la hipótesis nula: los errores son homocedásticos.")
else:
    print("Se rechaza la hipótesis nula: los errores no son homocedásticos.")

################################################################
```

donde ``X`` es la tabla formada por las columnas descriptivas con una columna de 1's añadida a la izquierda.

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Autocorrelación de los residuos: Test de Durbin-Watson.**

<font color = 'Black'>
<font size = 3>

Otra de las hipótesis que se deben verificar para que el **análisis de regresión** sea correcto es la incorrelación de los residuos.

La autocorrelación de los residuos puede ser de dos tipos:

* Positiva: un valor positivo (negativo) de un error genera una cadena de residuos positivos (negativos)

* Negativa: los residuos alternan signo.

Es decir, que la **correlación** entre los errores sea cero, por lo que son independientes entre sí. A menos que, las variables sean **gaussianas**. Si las variables son **gaussianas** la independencia y la **correlación** son lo mismo.

En general, si $U$ y $V$ son independientes, entonces su **correlación** $(U, V) = 0$. Pero al revés no: se puede tener **correlación** $(U, V) = 0$ pero no ser independientes.

Peeeeeero: si $U$ y $V$ son **gaussianas** con **correlación** $(U, V) = 0$, entonces sí son independientes.

Para comprobar que los residuos no presentan **correlación**, se puede aplicar el **Test de Durbin-Watson**.

Sean $\{e_i\}_{i=1}^n$ los residuos de la **regresión**. Tomemos $E_i$ y $E_{i-1}$ como las **variables aleatorias** de error y consideremos la **recta de regresión** de $E_i$ respecto de $E_{i-1}$. Esto es, consideremos el modelo $E_i=\beta_0+\beta_1E_{i-1}$.

Se plantea el contraste $$\left\{\begin{array}{l}\mathcal{H}_0:\beta_1=0\\\mathcal{H}_1:\beta_1\neq0\end{array}\right.$$ con el estadístico de contraste $$d=\frac{\sum_{i=2}^n(e_i-e_{i-1})^2}{\sum_{i=1}^ne_i^2}$$

El valor de este **estadístico** es aproximadamente $2(1-b_1)$ donde $b_1$ es una estimación de $\beta_1$. Si $\mathcal{H}_0$ es cierta, su distribución es la de una cierta combinación lineal de $\chi^2$.

La regla de decisión es

| Estadístico DW   | Interpretación                             |
|------------------|--------------------------------------------|
| DW < 1.5         | Posible autocorrelación positiva           |
| 1.5 <= DW <= 2.5 | No hay evidencia significativa de autocorrelación |
| DW > 2.5         | Posible autocorrelación negativa           |

---

<font color = 'DodgerBlue'>
<font size = 5>
    
**Diagnósticos del modelo: Aditividad y linealidad**

<font color = 'Black'>
<font size = 3>

Sea $i$ fija y perturbemos la variable $X_i$, digamos con $x_i+h$. Entonces

$$
\begin{array}{rcl}
\boldsymbol{Y|_{x_1,...,x_i+h,...x_k}-Y|_{x_1,...,x_i,...x_k}}&\boldsymbol{=}&\boldsymbol{\beta_0+\beta_1x_1+...+\beta_i(x_i+h)+...+\beta_kx_k-(\beta_0+\beta_1x_1+...+\beta_ix_i+...+\beta_kx_k)}\\
&\boldsymbol{=}&\boldsymbol{\beta_ih}
\end{array}
$$


Cuando se plantea un **modelo lineal**, se suponen implícitamente las condiciones siguientes:

* Aditividad: la variación en la expresión en **negrita** es independiente de las variables que no se perturbaron.

* Linealidad: la variación en la expresión en **negrita** es la misma independientemente de qué variable se perturbó.

Lo que nos quiere decir esto es que si tienes un **dato** perturbado y al restarle los **datos** originales, en esa diferencia solo se debe obtener el **dato** con la perturbación.

Este paso se debe hacer para evitar errores en los estudios estadísticos.

<font color = 'DodgerBlue'>
<font size = 5>
    
**Aditividad.**

<font color = 'Black'>
<font size = 3>

Podemos comprobar la aditividad con el **Test de Tukey** usando los llamados gráficos de residuos parciales. La idea principal es verificar que no haya interacción entre las variables independienes y así, cada una tendrá un efecto aditivo en el modelo.

Si existe la interacción, algunos de los términos cuadráticos tendrán peso en el modelo. Esta es la base del **Test de Tukey**.

Para aplicarlo, se siguen los siguientes pasos:

* Se obtienen los valores ajustados $\{\widehat{y_i}\}$ por la **regresión lineal** inicial.

* Se lleva a cabo una segunda **regresión lineal** incluyendo como nueva variable independiente los $\widehat{y_i}^2$. Sea $\beta$ el coeficiente de esta nueva variable.

* Se testea si la variable $\widehat{y}^2$ es significativa en la segunda regresión. Es decir, se realiza el contraste $$\left\{\begin{array}{l}\mathcal{H}_0:\beta=0\\\mathcal{H}_1:\beta\neq0\end{array}\right..$$

Si no podemos descartar $\mathcal{H}_0$, la variable de los $\widehat{y_i}^2$ no es significativa y el modelo es aditivo.

La regla de decisión es

$p$-valor|Decisión|Significado
:--|:--|:--
Pequeño|Rechazar $H_0$|Hay buena probabilidad de que el modelo **NO es aditivo**
Grande|Rechazar $H_1$|Hay buena probabilidad de que el modelo **SÍ es aditivo**

En el **Test de Tukey** se buscan p-valores grandes.

<font color = 'Brown'>
<font size = 4>
    
**Notas**

<font color = 'DarkRed'>
<font size = 3>

- Para poder confiar en nuestra **regresión lineal** más nos vale obtener p-valores grandes.
- En la **regresión lineal**, los errores deben ser **gaussianos**.

> ⚠️ **Nota importante: Si no se cumplen los diagnósticos de regresión (por ejemplo, si no hay gaussianidad en los residuos) simplemente no podemos confiar en los intervalos de confianza ni en la significancia de los coeficientes de la regresión.**

> ⚠️ **Nota importante: Recordar que los coeficientes son los números ($\beta_{0}$, $\beta{1}$, $\beta_{2}$, ..., $\beta_{k}$)**. Donde $Y = \beta_{0} + \beta{1} \ast X + \beta_{2} \ast X + ... + \beta_{k} \ast X$

- La **gaussianidad** se tiene que verificar en los errores cometidos, no en las columnas de la tabla.
- Si el $R^{2}$ ajustado es muy próximo a 1, es porque no tienes porqué preocuparte, simplemente no puedes confiar en los **intervalos de confianza**, porque los **intervalos de confianza** se construyen a partir del **teorema del límite central**.
- Si los errores no siguen una **distribución gaussianidad**, entonces los **intervalos de confianza** que se construyan para ellos no van a tener sentido. Los **intervalos de confianza** que conocemos solamente sirven para **gaussianidad**.
- Los errores cometidos deben tener **homocedasticidad**.
- Los errores cometidos son **variables aleatorias**  porque dependen de los **datos**. Por lo tanto, se les puede calcular su **varianza**. Lo que hay que ver es que todos tengan la misma **varianza**.
- En la **prueba de homocedasticidad**, más nos vale que en el **Test de Breuch-Pagan** salga un p-valor grande.
- En toda **prueba de hipótesis** siempre va haber un **estadístico** y un p-valor.